# Post 3 — Production LightGBM Experiments

Production-ready LightGBM notebook for the Favorita demand forecasting PRD. It uses the shared `src/demand` data pipeline and trains direct multi-step global LightGBM models from `build_supervised_frame`.

Primary supported runs:

- `feature_ablation`: PRD §4.5, core comparability slice only, five nested feature sets (`minimal`, `lag`, `rolling`, `promo`, `full`), point (log1p+L2) only, all configured test origins.
- `main_comparison`: PRD §5.1.2 / §5.1.3, configurable core or full-panel run for selected feature sets, with optional quantile boosters.
- `single`: one-off debugging or production dry run for any feature-set/scope/origin combination.

The first code cell is the control panel. Change only those knobs for normal use; the later cells derive paths, enforce PRD guardrails, run the experiment, and save all artifacts under `results/<OUTPUT_SUBDIR>/<RUN_NAME>/`.

This notebook is for labeled holdout benchmarking, not Kaggle submission generation. It intentionally ignores Favorita `test.csv` and evaluates only rolling origins whose forecast windows have observed targets in `train.csv`, so TabPFN-style models and LightGBM are compared on the same labeled test frame.

## 0. Control Panel

In [10]:
# -----------------------------------------------------------------------------
# Run preset
# -----------------------------------------------------------------------------
# "feature_ablation" = PRD §4.5: core slice, 5 feature sets, point (log1p+L2) only.
# "main_comparison" = configurable LightGBM main/scaling run.
# "single" = one feature set / custom smoke test.
EXPERIMENT_KIND = "main_comparison"
RUN_NAME = "lgbm_main_comparison_full"
OUTPUT_SUBDIR = "lgbm_engineered"

# -----------------------------------------------------------------------------
# Feature sets
# -----------------------------------------------------------------------------
ALL_FEATURE_SETS = ["minimal", "lag", "rolling", "promo", "full"]
FEATURE_SETS = ALL_FEATURE_SETS.copy()          # used by feature_ablation
SINGLE_FEATURE_SET = "full"                     # used by single
MAIN_FEATURE_SETS = ["minimal", "full"]         # common PRD main comparison pair

# -----------------------------------------------------------------------------
# Scope and origins
# -----------------------------------------------------------------------------
USE_CORE_SLICE = False                            # True = core shakedown; False = full LightGBM panel
WRITE_CORE_SLICE_CSV = True
N_TRAIN_ORIGINS = None                           # None = full PRD training grid; int = latest N origins
TEST_ORIGIN_INDICES = None                       # None = all test origins; e.g. [3] for one middle origin

# -----------------------------------------------------------------------------
# Models
# -----------------------------------------------------------------------------
TRAIN_POINT = True
TRAIN_QUANTILE_MODELS = True                     # needed for calibration/newsvendor main results
TUNED_PARAMS_PATH = None                         # e.g. "results/tuning/lgbm_point_full.json"

# -----------------------------------------------------------------------------
# Runtime mode
# -----------------------------------------------------------------------------
QUICK = False                                    # True caps n_estimators for iteration; False = config defaults
QUICK_N_ESTIMATORS = 400
QUICK_EARLY_STOPPING_ROUNDS = 25
MAX_FEATURE_SETS = None                          # optional local cap, e.g. 1 for smoke tests
FAIL_IF_OUTPUT_EXISTS = False
RANDOM_SEED = 42

# -----------------------------------------------------------------------------
# Artifact controls
# -----------------------------------------------------------------------------
SAVE_PARQUET = True
SAVE_MODELS = True                               # LightGBM text model files, not Python pickles
SAVE_FEATURE_IMPORTANCE = True

## 1. Imports, Config, and Guardrails

In [11]:
import copy
import json
import platform
import shutil
import subprocess
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from src.demand.config import load_config, resolve_path
from src.demand.data.features import FEATURE_SETS as VALID_FEATURE_SETS
from src.demand.data.load import build_panel, load_raw_data
from src.demand.data.splits import (
    build_core_slice,
    build_main_splits,
    family_regime,
    regime_horizon,
)
from src.demand.data.supervised_frame import build_supervised_frame
from src.demand.evaluation.metrics import (
    bias,
    interval_coverage,
    mae,
    mase,
    per_series_regime_mae,
    pinball_loss,
    rmse,
    smape,
)
from src.demand.models import lgbm_quantile, lgbm_point


def git_commit() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "no-git"


def load_tuned_params(path: str | None) -> dict | None:
    if not path:
        return None
    p = Path(path)
    if not p.is_absolute():
        p = Path.cwd() / p
    if not p.exists():
        raise FileNotFoundError(f"TUNED_PARAMS_PATH does not exist: {p}")
    raw_params = json.loads(p.read_text())
    objective = raw_params.get("_objective")
    if objective not in (None, "point", "quantile"):
        raise ValueError(f"TUNED_PARAMS_PATH must contain point (log1p+L2)/quantile params, got _objective={objective!r}")
    blocked = {"objective", "alpha", "log_target"}
    if blocked.intersection(raw_params):
        bad = sorted(blocked.intersection(raw_params))
        raise ValueError(f"TUNED_PARAMS_PATH should tune search params only, not objective params: {bad}")
    return {k: v for k, v in raw_params.items() if not k.startswith("_")}


def selected_feature_sets() -> list[str]:
    if EXPERIMENT_KIND == "feature_ablation":
        feats = list(FEATURE_SETS)
    elif EXPERIMENT_KIND == "main_comparison":
        feats = list(MAIN_FEATURE_SETS)
    elif EXPERIMENT_KIND == "single":
        feats = [SINGLE_FEATURE_SET]
    else:
        raise ValueError("EXPERIMENT_KIND must be one of: feature_ablation, main_comparison, single")
    if MAX_FEATURE_SETS is not None:
        feats = feats[: int(MAX_FEATURE_SETS)]
    bad = [f for f in feats if f not in VALID_FEATURE_SETS]
    if bad:
        raise ValueError(f"Unknown feature sets: {bad}; valid={VALID_FEATURE_SETS}")
    return feats


def apply_guardrails(feature_sets: list[str]) -> None:
    if not TRAIN_POINT:
        raise ValueError("This notebook requires TRAIN_POINT=True; quantiles are optional add-ons.")
    if EXPERIMENT_KIND == "feature_ablation":
        if feature_sets != list(ALL_FEATURE_SETS)[: len(feature_sets)]:
            raise ValueError("PRD §4.5 ablation should run the nested order: minimal, lag, rolling, promo, full.")
        if not USE_CORE_SLICE:
            raise ValueError("PRD §4.5 ablation scope is core comparability slice only; set USE_CORE_SLICE=True.")
        if TRAIN_QUANTILE_MODELS:
            raise ValueError("PRD §4.5 ablation uses point (log1p+L2) only; set TRAIN_QUANTILE_MODELS=False.")
    if QUICK:
        print("QUICK=True: capping LightGBM rounds for iteration. Do not use these artifacts for final claims.")


cfg = load_config("default.yaml")
cfg = copy.deepcopy(cfg)  # detach from any shared mutable references
cfg["seed"] = RANDOM_SEED
if QUICK:
    cfg["lgbm"]["defaults"]["n_estimators"] = QUICK_N_ESTIMATORS
    cfg["lgbm"]["search_space"]["early_stopping_rounds"] = QUICK_EARLY_STOPPING_ROUNDS

splits = build_main_splits(cfg)
feature_sets_to_run = selected_feature_sets()
apply_guardrails(feature_sets_to_run)
tuned_params = load_tuned_params(TUNED_PARAMS_PATH)

out_root = resolve_path(cfg, "results_dir") / OUTPUT_SUBDIR / RUN_NAME
model_dir = out_root / "models"
if FAIL_IF_OUTPUT_EXISTS and out_root.exists():
    raise FileExistsError(f"Output directory already exists: {out_root}")
out_root.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

all_test_origins = list(splits.test_origins)
if TEST_ORIGIN_INDICES is None:
    test_origins = all_test_origins
else:
    test_origins = [all_test_origins[i] for i in TEST_ORIGIN_INDICES]
training_origins = list(splits.training_origins)
if N_TRAIN_ORIGINS is not None and N_TRAIN_ORIGINS < len(training_origins):
    training_origins = training_origins[-int(N_TRAIN_ORIGINS):]
validation_origins = list(splits.validation_origins)

print(f"experiment:      {EXPERIMENT_KIND}")
print(f"run:             {RUN_NAME}")
print(f"output:          {out_root}")
print(f"feature sets:    {feature_sets_to_run}")
print(f"core slice:      {USE_CORE_SLICE}")
print(f"train origins:   {len(training_origins)} ({training_origins[0].date()} -> {training_origins[-1].date()})")
print(f"val origins:     {len(validation_origins)}")
print(f"test origins:    {[o.date().isoformat() for o in test_origins]}")
print(f"quantiles:       {TRAIN_QUANTILE_MODELS}")
print(f"quick:           {QUICK}")
print(f"git commit:      {git_commit()}")

experiment:      main_comparison
run:             lgbm_main_comparison_full
output:          /Users/alexruppelt/Documents/Projects/tabpfn_analysis/results/lgbm_engineered/lgbm_main_comparison_full
feature sets:    ['minimal', 'full']
core slice:      False
train origins:   206 (2013-01-28 -> 2017-01-02)
val origins:     8
test origins:    ['2017-04-30', '2017-05-14', '2017-05-28', '2017-06-11', '2017-06-25', '2017-07-09', '2017-07-17']
quantiles:       True
quick:           False
git commit:      c58e1c9


## 2. Load Data and Select Scope

In [12]:
t0 = time.time()
raw = load_raw_data(cfg)
# Benchmarking uses labeled holdout windows from train.csv only. Favorita test.csv
# is intentionally excluded because it has no target and is only needed for Kaggle submissions.
panel = build_panel(raw, include_test=False)
print(f"loaded panel in {time.time() - t0:.1f}s")
print(f"panel rows:   {len(panel):,}")
print(f"date range:   {panel['date'].min().date()} -> {panel['date'].max().date()}")
print(f"series:       {panel[['store_nbr', 'family']].drop_duplicates().shape[0]:,}")

if USE_CORE_SLICE:
    series_filter = build_core_slice(panel, raw.stores, cfg, write_csv=WRITE_CORE_SLICE_CSV)
    print(f"core slice:   {len(series_filter):,} series")
    print(f"families:     {series_filter['family'].nunique()} / {panel['family'].nunique()}")
else:
    series_filter = None
    print("scope:        full panel")

panel.head(3)

Info: train.csv omits 7,128 Dec 25 closure rows; build_panel() will zero-fill them.
loaded panel in 3.7s
panel rows:   3,008,016
date range:   2013-01-01 -> 2017-08-15
series:       1,782
scope:        full panel


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,oil_price,transactions
0,0.0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13.0,93.14,NaN
1,1782.0,2013-01-02,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13.0,93.14,2111.0
2,3564.0,2013-01-03,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13.0,92.97,1833.0


## 3. Metric Helpers

The helpers keep the evaluation consistent across a single full-feature run and the §4.5 five-level ablation. Non-perishable metrics aggregate each `(origin, store_nbr, family)` over the configured horizon window before scoring, matching PRD §7.3.

In [13]:
def actuals_for_forecasts(panel: pd.DataFrame, forecasts: pd.DataFrame) -> pd.DataFrame:
    dates = forecasts["date"].drop_duplicates()
    keys = forecasts[["store_nbr", "family"]].drop_duplicates()
    actuals = panel.merge(keys, on=["store_nbr", "family"], how="inner")
    actuals = actuals[actuals["date"].isin(dates)]
    return actuals[["date", "store_nbr", "family", "sales"]]


def join_actuals(panel: pd.DataFrame, forecasts: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    actuals = actuals_for_forecasts(panel, forecasts)
    joined = forecasts.merge(actuals, on=["date", "store_nbr", "family"], how="inner")
    joined["regime"] = joined["family"].map(lambda f: family_regime(f, cfg))
    return joined


def compute_regime_metrics(joined: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    rows = []
    for regime in ("perishable", "non_perishable"):
        lo, hi = regime_horizon(regime, cfg)
        sub = joined[(joined["regime"] == regime) & (joined["horizon_offset"].between(lo, hi))]
        if sub.empty:
            continue
        if regime == "perishable":
            eval_frame = sub.copy()
            y = eval_frame["sales"].to_numpy()
            yhat = eval_frame["pred"].to_numpy()
            n_rows = len(eval_frame)
        else:
            eval_frame = sub.groupby(["origin", "store_nbr", "family"], as_index=False).agg(
                sales=("sales", "sum"),
                pred=("pred", "sum"),
            )
            y = eval_frame["sales"].to_numpy()
            yhat = eval_frame["pred"].to_numpy()
            n_rows = len(eval_frame)

        row = {
            "feature_set": feature_set,
            "regime": regime,
            "n_rows": int(n_rows),
            "mae": mae(y, yhat),
            "rmse": rmse(y, yhat),
            "smape": smape(y, yhat),
            "bias": bias(y, yhat),
        }
        if regime == "perishable" and {"q10", "q50", "q90"}.issubset(sub.columns):
            # Quantile sums are not additive, so only report quantile diagnostics
            # for the perishable row-level decision target.
            row.update({
                "pinball_q10": pinball_loss(sub["sales"], sub["q10"], 0.1),
                "pinball_q50": pinball_loss(sub["sales"], sub["q50"], 0.5),
                "pinball_q90": pinball_loss(sub["sales"], sub["q90"], 0.9),
                "coverage_80": interval_coverage(sub["sales"], sub["q10"], sub["q90"]),
            })
        rows.append(row)
    return pd.DataFrame(rows)


def compute_mase_by_origin(joined: pd.DataFrame, panel: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    training_history = panel[panel["date"] <= splits.train_end]
    rows = []
    for (origin, store, fam), g in joined.groupby(["origin", "store_nbr", "family"], sort=False):
        hist = training_history.loc[
            (training_history["store_nbr"] == store) & (training_history["family"] == fam),
            "sales",
        ].to_numpy()
        value = mase(g["sales"].to_numpy(), g["pred"].to_numpy(), hist, m=7)
        if np.isnan(value):
            continue
        rows.append({
            "feature_set": feature_set,
            "origin": origin,
            "store_nbr": store,
            "family": fam,
            "series_id": f"{store}__{fam}",
            "regime": family_regime(fam, cfg),
            "mase": value,
        })
    return pd.DataFrame(rows)


def compute_per_series_mae(forecasts: pd.DataFrame, actuals: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    frames = []
    for regime in ("perishable", "non_perishable"):
        lo, hi = regime_horizon(regime, cfg)
        fam_in_regime = forecasts["family"].map(lambda f: family_regime(f, cfg)) == regime
        fc_subset = forecasts[fam_in_regime]
        if fc_subset.empty:
            continue
        part = per_series_regime_mae(fc_subset, actuals, regime, (lo, hi), pred_col="pred")
        part.insert(0, "feature_set", feature_set)
        frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def summarize_mase(mase_df: pd.DataFrame) -> pd.DataFrame:
    if mase_df.empty:
        return pd.DataFrame()
    return (
        mase_df.groupby(["feature_set", "regime"], as_index=False)["mase"]
        .agg(["mean", "median", "count"])
        .reset_index()
    )

## 4. Training and Forecast Function

In [14]:
def build_frames_for_feature_set(feature_set: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    t0 = time.time()
    frame_train = build_supervised_frame(
        panel,
        origins=training_origins,
        horizon=cfg["horizon_days"],
        feature_set=feature_set,
        config=cfg,
        mode="train",
        holidays=raw.holidays,
        stores=raw.stores,
        series_filter=series_filter,
    )
    frame_val = build_supervised_frame(
        panel,
        origins=validation_origins,
        horizon=cfg["horizon_days"],
        feature_set=feature_set,
        config=cfg,
        mode="train",
        holidays=raw.holidays,
        stores=raw.stores,
        series_filter=series_filter,
    )
    frame_forecast = build_supervised_frame(
        panel,
        origins=test_origins,
        horizon=cfg["horizon_days"],
        feature_set=feature_set,
        config=cfg,
        mode="forecast",
        holidays=raw.holidays,
        stores=raw.stores,
        series_filter=series_filter,
    )
    print(
        f"[{feature_set}] frames in {time.time() - t0:.1f}s | "
        f"train={len(frame_train):,} val={len(frame_val):,} forecast={len(frame_forecast):,}"
    )
    if frame_train.empty or frame_forecast.empty:
        raise ValueError(f"Empty supervised frame for feature_set={feature_set}")
    return frame_train, frame_val, frame_forecast


def save_booster(artifact, feature_set: str, suffix: str) -> str | None:
    if not SAVE_MODELS:
        return None
    model_path = model_dir / f"{feature_set}_{suffix}.txt"
    artifact.booster.save_model(str(model_path))
    metadata = {
        "name": artifact.name,
        "objective": artifact.objective,
        "feature_set": artifact.feature_set,
        "feature_cols": artifact.feature_cols,
        "categorical_cols": artifact.categorical_cols,
        "alpha": artifact.alpha,
        "log_target": artifact.log_target,
        "params": artifact.params,
    }
    model_path.with_suffix(".json").write_text(json.dumps(metadata, indent=2, default=str))
    return str(model_path.relative_to(out_root))


def run_feature_set(feature_set: str) -> dict:
    feature_start = time.time()
    frame_train, frame_val, frame_forecast = build_frames_for_feature_set(feature_set)

    t0 = time.time()
    point_artifact = lgbm_point.train(
        frame_train=frame_train,
        frame_val=frame_val,
        feature_set=feature_set,
        cfg=cfg,
        tuned_params=tuned_params,
    )
    point_train_seconds = time.time() - t0
    point_model_path = save_booster(point_artifact, feature_set, "point")
    print(f"[{feature_set}] point (log1p+L2) trained in {point_train_seconds:.1f}s; best_iter={point_artifact.booster.best_iteration}")

    quantile_artifacts = []
    quantile_train_seconds = 0.0
    quantile_model_paths = []
    if TRAIN_QUANTILE_MODELS:
        t0 = time.time()
        quantile_artifacts = lgbm_quantile.train_alphas(
            frame_train=frame_train,
            frame_val=frame_val,
            feature_set=feature_set,
            cfg=cfg,
            tuned_params=tuned_params,
        )
        quantile_train_seconds = time.time() - t0
        for art in quantile_artifacts:
            quantile_model_paths.append(save_booster(art, feature_set, f"q{int(round(art.alpha * 100))}"))
        print(f"[{feature_set}] quantiles trained in {quantile_train_seconds:.1f}s")

    t0 = time.time()
    forecasts = lgbm_point.predict(point_artifact, frame_forecast)
    if quantile_artifacts:
        qframe = lgbm_quantile.predict_quantiles(quantile_artifacts, frame_forecast, cfg)
        key = ["origin", "date", "horizon_offset", "store_nbr", "family", "series_id"]
        forecasts = forecasts.merge(qframe, on=key, how="left")
    forecast_seconds = time.time() - t0
    forecasts.insert(0, "feature_set", feature_set)

    actuals = actuals_for_forecasts(panel, forecasts)
    joined = join_actuals(panel, forecasts, cfg)
    metrics_df = compute_regime_metrics(joined, feature_set)
    mase_df = compute_mase_by_origin(joined, panel, feature_set)
    per_series_df = compute_per_series_mae(forecasts, actuals, feature_set)

    if SAVE_PARQUET:
        forecasts.to_parquet(out_root / f"forecasts_{feature_set}.parquet", index=False)
        joined.to_parquet(out_root / f"joined_{feature_set}.parquet", index=False)
        metrics_df.to_parquet(out_root / f"metrics_{feature_set}.parquet", index=False)
        if not mase_df.empty:
            mase_df.to_parquet(out_root / f"mase_per_series_{feature_set}.parquet", index=False)
        if not per_series_df.empty:
            per_series_df.to_parquet(out_root / f"per_series_mae_{feature_set}.parquet", index=False)

    feature_importance_path = None
    if SAVE_FEATURE_IMPORTANCE:
        imp = (
            pd.DataFrame({
                "feature_set": feature_set,
                "feature": point_artifact.booster.feature_name(),
                "gain": point_artifact.booster.feature_importance(importance_type="gain"),
                "split": point_artifact.booster.feature_importance(importance_type="split"),
            })
            .sort_values("gain", ascending=False)
            .reset_index(drop=True)
        )
        feature_importance_path = out_root / f"feature_importance_{feature_set}.parquet"
        imp.to_parquet(feature_importance_path, index=False)

    elapsed = time.time() - feature_start
    rows_predicted = int(len(forecasts))
    return {
        "feature_set": feature_set,
        "frames": {
            "n_train_rows": int(len(frame_train)),
            "n_val_rows": int(len(frame_val)),
            "n_forecast_rows": int(len(frame_forecast)),
            "n_series": int(frame_forecast[["store_nbr", "family"]].drop_duplicates().shape[0]),
            "n_test_origins": int(len(test_origins)),
        },
        "timing": {
            "point_train_seconds": point_train_seconds,
            "quantile_train_seconds": quantile_train_seconds,
            "forecast_seconds": forecast_seconds,
            "total_seconds": elapsed,
            "predictions_per_second": rows_predicted / forecast_seconds if forecast_seconds > 0 else None,
        },
        "models": {
            "point": point_model_path,
            "quantiles": quantile_model_paths,
        },
        "metrics": metrics_df,
        "mase": mase_df,
        "per_series": per_series_df,
        "forecasts": forecasts,
    }

## 5. Run Experiment

In [15]:
run_start = time.time()
results = []
for feature_set in feature_sets_to_run:
    print("=" * 80)
    print(f"Running feature_set={feature_set}")
    results.append(run_feature_set(feature_set))

print("=" * 80)
print(f"finished {len(results)} feature-set run(s) in {(time.time() - run_start) / 60:.1f} min")

Running feature_set=minimal
[minimal] frames in 199.7s | train=10,278,576 val=399,168 forecast=349,272
[minimal] point (log1p+L2) trained in 149.5s; best_iter=1999
[minimal] quantiles trained in 534.0s
Running feature_set=full
[full] frames in 1091.9s | train=10,278,576 val=399,168 forecast=349,272
[full] point (log1p+L2) trained in 54.2s; best_iter=355
[full] quantiles trained in 254.6s
finished 2 feature-set run(s) in 64.1 min


## 6. Combined Results and Runtime Tables

In [16]:
def concat_nonempty(frames: list[pd.DataFrame]) -> pd.DataFrame:
    frames = [f for f in frames if f is not None and not f.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


all_forecasts = concat_nonempty([r["forecasts"] for r in results])
all_metrics = concat_nonempty([r["metrics"] for r in results])
all_mase = concat_nonempty([r["mase"] for r in results])
all_per_series = concat_nonempty([r["per_series"] for r in results])
runtime_rows = []
for r in results:
    runtime_rows.append({
        "model_name": "lgbm_point",
        "feature_set": r["feature_set"],
        "origin": "all_selected",
        "n_series": r["frames"]["n_series"],
        "n_test_origins": r["frames"]["n_test_origins"],
        "n_predictions": r["frames"]["n_forecast_rows"],
        **r["timing"],
        "hardware_spec": platform.platform(),
        "git_commit": git_commit(),
        "timestamp": pd.Timestamp.now("UTC").isoformat(),
    })
runtime_df = pd.DataFrame(runtime_rows)
mase_summary = summarize_mase(all_mase)

if SAVE_PARQUET:
    all_forecasts.to_parquet(out_root / "forecasts_all.parquet", index=False)
    all_metrics.to_parquet(out_root / "metrics_all.parquet", index=False)
    if not all_mase.empty:
        all_mase.to_parquet(out_root / "mase_per_series_all.parquet", index=False)
    if not all_per_series.empty:
        all_per_series.to_parquet(out_root / "per_series_mae_all.parquet", index=False)
    runtime_df.to_parquet(out_root / "runtime_log.parquet", index=False)
    if not mase_summary.empty:
        mase_summary.to_parquet(out_root / "mase_summary.parquet", index=False)

print("Metrics")
display(all_metrics.sort_values(["feature_set", "regime"]))
print("MASE summary")
display(mase_summary.sort_values(["feature_set", "regime"]) if not mase_summary.empty else mase_summary)
print("Runtime")
display(runtime_df)

Metrics


,feature_set,regime,n_rows,mae,rmse,smape,bias,pinball_q10,pinball_q50,pinball_q90,coverage_80
3,full,non_perishable,9072,991.282854,4067.182157,0.405266,-0.050329,NaN,NaN,NaN,NaN
2,full,perishable,23814,100.576420,295.005090,0.222091,0.013188,24.612534,52.129106,19.494938,0.786932
1,minimal,non_perishable,9072,1194.416580,5628.286744,0.490678,-0.066961,NaN,NaN,NaN,NaN
0,minimal,perishable,23814,144.652767,447.335592,0.295208,0.021785,24.173445,61.723024,44.839511,0.766104


MASE summary


,index,feature_set,regime,mean,median,count
0,0,full,non_perishable,1.427811,1.030521,8533
1,1,full,perishable,1.102629,0.925159,3339
2,2,minimal,non_perishable,1.778004,1.149956,8533
3,3,minimal,perishable,1.412183,1.037567,3339


Runtime


,model_name,feature_set,origin,n_series,n_test_origins,n_predictions,point_train_seconds,quantile_train_seconds,forecast_seconds,total_seconds,predictions_per_second,hardware_spec,git_commit,timestamp
0,lgbm_point,minimal,all_selected,1782,7,349272,149.482192,534.009616,9.692105,1671.458939,36036.753422,macOS-26.5.1-arm64-arm-64bit,c58e1c9,2026-06-17T13:31:37.282065+00:00
1,lgbm_point,full,all_selected,1782,7,349272,54.229615,254.606742,3.119698,2174.609561,111956.988999,macOS-26.5.1-arm64-arm-64bit,c58e1c9,2026-06-17T13:31:37.298341+00:00


## 7. Save Run Summary

In [20]:
summary = {
    "experiment_kind": EXPERIMENT_KIND,
    "run_name": RUN_NAME,
    "output_dir": str(out_root),
    "feature_sets": feature_sets_to_run,
    "use_core_slice": USE_CORE_SLICE,
    "quick": QUICK,
    "train_quantile_models": TRAIN_QUANTILE_MODELS,
    "tuned_params_path": TUNED_PARAMS_PATH,
    "n_train_origins": len(training_origins),
    "training_origins": [o.date().isoformat() for o in training_origins],
    "validation_origins": [o.date().isoformat() for o in validation_origins],
    "test_origins": [o.date().isoformat() for o in test_origins],
    "horizon_days": cfg["horizon_days"],
    "lgbm_defaults": cfg["lgbm"]["defaults"],
    "lgbm_early_stopping_rounds": cfg["lgbm"]["search_space"].get("early_stopping_rounds"),
    "frames": {r["feature_set"]: r["frames"] for r in results},
    "timing": {r["feature_set"]: r["timing"] for r in results},
    "metrics": all_metrics.to_dict(orient="records"),
    "runtime_log_path": "runtime_log.parquet",
    "git_commit": git_commit(),
    "created_at": pd.Timestamp.now("UTC").isoformat(),
}
(out_root / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(f"wrote run summary to {out_root / 'run_summary.json'}")
print("artifacts:")
for p in sorted(out_root.glob("*")):
    if p.is_file():
        print(f"  {p.name}")

wrote run summary to /Users/alexruppelt/Documents/Projects/tabpfn_analysis/results/lgbm_engineered/lgbm_main_comparison_full/run_summary.json
artifacts:
  feature_importance_full.parquet
  feature_importance_minimal.parquet
  forecasts_all.parquet
  forecasts_full.parquet
  forecasts_minimal.parquet
  joined_full.parquet
  joined_minimal.parquet
  mase_per_series_all.parquet
  mase_per_series_full.parquet
  mase_per_series_minimal.parquet
  mase_summary.parquet
  metrics_all.parquet
  metrics_full.parquet
  metrics_minimal.parquet
  per_series_mae_all.parquet
  per_series_mae_full.parquet
  per_series_mae_minimal.parquet
  run_summary.json
  runtime_log.parquet


## 8. Quick Artifact Validation

In [21]:
expected_forecast_rows = len(test_origins) * cfg["horizon_days"]
if USE_CORE_SLICE:
    expected_forecast_rows *= len(series_filter)
else:
    expected_forecast_rows *= panel[["store_nbr", "family"]].drop_duplicates().shape[0]

checks = []
for feature_set in feature_sets_to_run:
    sub = all_forecasts[all_forecasts["feature_set"] == feature_set]
    checks.append({
        "feature_set": feature_set,
        "rows": len(sub),
        "expected_rows": expected_forecast_rows,
        "row_count_ok": len(sub) == expected_forecast_rows,
        "prediction_nulls": int(sub["pred"].isna().sum()),
        "negative_predictions": int((sub["pred"] < 0).sum()),
        "series": int(sub[["store_nbr", "family"]].drop_duplicates().shape[0]),
        "origins": int(sub["origin"].nunique()),
        "horizon_min": int(sub["horizon_offset"].min()),
        "horizon_max": int(sub["horizon_offset"].max()),
    })
    if {"q10", "q50", "q90"}.issubset(sub.columns):
        checks[-1]["quantile_order_violations"] = int(
            ((sub["q10"] > sub["q50"]) | (sub["q50"] > sub["q90"])).sum()
        )
validation_df = pd.DataFrame(checks)
display(validation_df)

bad = validation_df[
    (~validation_df["row_count_ok"])
    | (validation_df["prediction_nulls"] > 0)
    | (validation_df["negative_predictions"] > 0)
    | (validation_df["horizon_min"] != 1)
    | (validation_df["horizon_max"] != cfg["horizon_days"])
]
if not bad.empty:
    raise AssertionError(f"Artifact validation failed:\n{bad}")
print("validation passed")

,feature_set,rows,expected_rows,row_count_ok,prediction_nulls,negative_predictions,series,origins,horizon_min,horizon_max,quantile_order_violations
0,minimal,349272,349272,True,0,0,1782,7,1,28,0
1,full,349272,349272,True,0,0,1782,7,1,28,0


validation passed


## 9. Feature Importance Peek

In [22]:
importance_frames = []
for feature_set in feature_sets_to_run:
    p = out_root / f"feature_importance_{feature_set}.parquet"
    if p.exists():
        importance_frames.append(pd.read_parquet(p).head(15))
feature_importance_top = pd.concat(importance_frames, ignore_index=True) if importance_frames else pd.DataFrame()
feature_importance_top

,feature_set,feature,gain,split
0,minimal,family,3.822478e+08,27563
1,minimal,store_nbr,7.122643e+07,35481
2,minimal,onpromotion,6.621740e+07,4674
3,minimal,year,2.053178e+07,7814
4,minimal,oil_price,2.026208e+07,10482
5,minimal,month,1.669865e+07,13863
6,minimal,city,6.492769e+06,3811
7,minimal,cluster,6.410220e+06,2884
8,minimal,state,4.043951e+06,582
9,minimal,day_of_week,3.522160e+06,6564
